In [1]:
import glob
import numpy as np
import time
from PIL import Image
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchmetrics.classification import MulticlassConfusionMatrix
from torch.utils.data import Dataset, DataLoader

In [2]:
CLASSES = [ 
    'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting', 'Cyclist', 'Tram', 'Misc'
]

colors = ['#ffffff','#3a3b7b','#6a6ecf','#8ca351','#fff100','#ff00ff','#833c39','#e598a0']

RESIZE_TO = 640 

ANCHORS = [
    # LARGE OBJECT SCALE (S=20)
    [
        [0.046875 , 0.0953125],
        [0.096875 , 0.0703125],
        [0.0921875, 0.1859375]
    ],
    # SMALL OBJECT SCALE (S=40)
    [
        [0.015625 , 0.0203125],
        [0.025    , 0.046875 ],
        [0.0515625, 0.0359375]
    ],
]


S = [RESIZE_TO // 32, RESIZE_TO // 16]

NUM_CLASSES = len(CLASSES) 
NUM_WORKERS = 4
BATCH_SIZE = 5 * 2
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4 
PIN_MEMORY = True
AMP = True
EPOCHS = 100

CHECKPOINT_FILE_42 = "./checkpoints/best_full_abl_std_42.pth.tar"
CHECKPOINT_FILE_123 = "./checkpoints/best_full_abl_std_123.pth.tar"
CHECKPOINT_FILE_999 = "./checkpoints/best_full_abl_std_999.pth.tar"

BASE_DATASET_PATH = '../../datasets/KITTI/dataset'
test_images_path = f'{BASE_DATASET_PATH}/test/images/'

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
""" 
Information about architecture config:
"B" indicating a residual block
"S" is for scale prediction block
"U" is for upsampling the feature map
"""
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 4],
    (128, 3, 2),
    ["B", 6],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 6],
    (512, 1, 1),
    (1024, 3, 1),

    # ONLY ONE DETECTION HEAD
    "S"
]

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn_act = bn_act

    def forward(self, x):
        if self.use_bn_act:
            return self.leaky(self.bn(self.conv(x)))
        else:
            return self.conv(x)


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()
        for repeat in range(num_repeats):
            self.layers += [
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            ]

        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)

        return x


class ScalePrediction(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(
                2 * in_channels, 3 * (num_classes + 5), bn_act=False, kernel_size=1
            ),
        )
        
        self.num_classes = num_classes

    def forward(self, x):
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )

class SiStNet(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []
        route_connections = []
        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()

        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels
        
        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0,
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(in_channels, num_repeats=num_repeats))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels=in_channels // 2, num_classes=self.num_classes),
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2))
                    in_channels = in_channels * 3

        return layers

if __name__ == "__main__": 
    model = SiStNet(num_classes=NUM_CLASSES)

In [5]:

test_transforms = A.Compose(
    [
        A.LongestMaxSize(max_size=RESIZE_TO),
        A.PadIfNeeded(
            min_height=RESIZE_TO, min_width=RESIZE_TO, border_mode=cv2.BORDER_CONSTANT
        ),
        A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255,),
        ToTensorV2(),
    ],
)

In [6]:
model = SiStNet(num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.cuda.amp.GradScaler(
    enabled=AMP,
    init_scale=2**12
)

# ---------------------------
# 3. CHECKPOINT LOAD
# ---------------------------
def load_model(checkpoint_path, model, optimizer, scheduler, scaler):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)    
    
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    if scheduler and checkpoint["scheduler"]:
        scheduler.load_state_dict(checkpoint["scheduler"])

    if scaler and checkpoint["scaler"]:
        scaler.load_state_dict(checkpoint["scaler"])

    start_epoch = checkpoint["epoch"] + 1
    best_map = checkpoint["best_map"]

    seed = checkpoint["seed"]

    #torch_rng_state = torch.set_rng_state(checkpoint["torch_rng_state"])
    #cuda_rng_state = torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])
    #numpy_rng_state = np.random.set_state(checkpoint["numpy_rng_state"])
    #python_rng_state = random.setstate(checkpoint["python_rng_state"])

    print("Full Checkpoint loaded!")
    
    return start_epoch, best_map, seed
    

# ---------------------------
# 4. INFERENCE DATASET (LABELSIZ)
# ---------------------------
class InferenceDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.images = glob.glob(f"{img_dir}/*")
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = np.array(Image.open(img_path).convert("RGB"))

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, img_path


In [7]:
import time
import numpy as np
import torch

# ---------------------------
# CUDA OPTIMIZATION
# ---------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False


# ---------------------------
# PER-IMAGE BENCHMARK (METHOD 1)
# ---------------------------
def run_per_image_benchmark(model, loader, device):
    model.eval()

    latencies = []
    total_images = 0

    # ---------------- WARMUP ----------------
    with torch.no_grad():
        for i in range(10):
            img = next(iter(loader))[0][0].unsqueeze(0).to(device)
            _ = model(img)

    if device == "cuda":
        torch.cuda.synchronize()

    # ---------------- TIMED ----------------
    with torch.no_grad():
        for images, _ in loader:

            for img in images:
                img = img.unsqueeze(0).to(device)

                if device == "cuda":
                    torch.cuda.synchronize()

                start = time.perf_counter()
                _ = model(img)

                if device == "cuda":
                    torch.cuda.synchronize()

                end = time.perf_counter()

                latency = (end - start) * 1000
                latencies.append(latency)
                total_images += 1

                print(f"Image | {latency:.2f} ms | {1000/latency:.2f} FPS")

    total_time = sum(latencies) / 1000

    print("\n====================")
    print("FINAL RESULTS")
    print("====================")
    print(f"Avg Latency: {np.mean(latencies):.2f} ms")
    print(f"Throughput FPS: {total_images / total_time:.2f}")

    return total_images / total_time, np.mean(latencies)

In [8]:
# ---------------------------
# 6. DATASET + DATALOADER
# ---------------------------
test_dataset = InferenceDataset(
    img_dir=test_images_path,
    transform=test_transforms
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ---------------------------
# 7. ANCHORS
# ---------------------------
scaled_anchors = [
    torch.tensor(ANCHORS[i], device=DEVICE, dtype=torch.float32) * float(S[i])
    for i in range(len(S))
]



In [9]:
def run_batch_benchmark(model, loader, device):
    model.eval()

    total_time = 0.0
    total_images = 0

    # ---------------- WARMUP ----------------
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            images = images.to(device)
            _ = model(images)
            if i == 5:
                break

    if device == "cuda":
        torch.cuda.synchronize()

    # ---------------- TIMED ----------------
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):

            images = images.to(device)
            batch_size = images.shape[0]

            if device == "cuda":
                torch.cuda.synchronize()

            start = time.perf_counter()
            _ = model(images)
            if device == "cuda":
                torch.cuda.synchronize()
            end = time.perf_counter()

            batch_time = end - start

            total_time += batch_time
            total_images += batch_size

            batch_fps = batch_size / batch_time
            latency = (batch_time / batch_size) * 1000

            print(
                f"Batch {batch_idx} | "
                f"Batch FPS: {batch_fps:.2f} | "
                f"Latency: {latency:.2f} ms/img"
            )

    avg_fps = total_images / total_time
    avg_latency = (total_time / total_images) * 1000

    print("\n====================")
    print("FINAL BATCH RESULTS")
    print("====================")
    print(f"Avg FPS (throughput): {avg_fps:.2f}")
    print(f"Avg Latency: {avg_latency:.2f} ms/image")

    return avg_fps, avg_latency

In [10]:

start_epoch, best_map, seed = load_model(CHECKPOINT_FILE_42, model, optimizer, scheduler, scaler)

# ---------------------------
# 8. RUN BENCHMARK
# ---------------------------

avg_fps, avg_latency = run_batch_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)


avg_fps, avg_latency = run_per_image_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)


Full Checkpoint loaded!
Batch 0 | Batch FPS: 24.72 | Latency: 40.45 ms/img
Batch 1 | Batch FPS: 24.59 | Latency: 40.66 ms/img

FINAL BATCH RESULTS
Avg FPS (throughput): 24.66
Avg Latency: 40.56 ms/image
Image | 68.07 ms | 14.69 FPS
Image | 65.43 ms | 15.28 FPS
Image | 64.21 ms | 15.57 FPS
Image | 61.41 ms | 16.28 FPS
Image | 57.10 ms | 17.51 FPS
Image | 52.74 ms | 18.96 FPS
Image | 52.78 ms | 18.94 FPS
Image | 52.63 ms | 19.00 FPS
Image | 53.18 ms | 18.80 FPS
Image | 54.01 ms | 18.51 FPS
Image | 52.87 ms | 18.91 FPS
Image | 52.96 ms | 18.88 FPS
Image | 53.23 ms | 18.79 FPS
Image | 54.31 ms | 18.41 FPS
Image | 52.18 ms | 19.17 FPS
Image | 52.62 ms | 19.00 FPS
Image | 52.88 ms | 18.91 FPS
Image | 53.79 ms | 18.59 FPS
Image | 52.65 ms | 18.99 FPS
Image | 52.81 ms | 18.93 FPS

FINAL RESULTS
Avg Latency: 55.59 ms
Throughput FPS: 17.99


In [11]:

start_epoch, best_map, seed = load_model(CHECKPOINT_FILE_123, model, optimizer, scheduler, scaler)

avg_fps, avg_latency = run_batch_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

# ---------------------------
# 8. RUN BENCHMARK
# ---------------------------
avg_fps, avg_latency = run_per_image_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

Full Checkpoint loaded!
Batch 0 | Batch FPS: 24.56 | Latency: 40.72 ms/img
Batch 1 | Batch FPS: 24.49 | Latency: 40.83 ms/img

FINAL BATCH RESULTS
Avg FPS (throughput): 24.53
Avg Latency: 40.77 ms/image
Image | 68.08 ms | 14.69 FPS
Image | 68.25 ms | 14.65 FPS
Image | 67.80 ms | 14.75 FPS
Image | 55.07 ms | 18.16 FPS
Image | 53.38 ms | 18.73 FPS
Image | 53.47 ms | 18.70 FPS
Image | 52.51 ms | 19.04 FPS
Image | 52.67 ms | 18.99 FPS
Image | 52.96 ms | 18.88 FPS
Image | 53.97 ms | 18.53 FPS
Image | 52.62 ms | 19.00 FPS
Image | 52.92 ms | 18.90 FPS
Image | 52.71 ms | 18.97 FPS
Image | 54.27 ms | 18.43 FPS
Image | 53.78 ms | 18.60 FPS
Image | 53.49 ms | 18.69 FPS
Image | 53.02 ms | 18.86 FPS
Image | 54.31 ms | 18.41 FPS
Image | 52.70 ms | 18.98 FPS
Image | 52.68 ms | 18.98 FPS

FINAL RESULTS
Avg Latency: 55.53 ms
Throughput FPS: 18.01


In [12]:
start_epoch, best_map, seed = load_model(CHECKPOINT_FILE_999, model, optimizer, scheduler, scaler)

avg_fps, avg_latency = run_batch_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

# ---------------------------
# 8. RUN BENCHMARK
# ---------------------------
avg_fps, avg_latency = run_per_image_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

Full Checkpoint loaded!
Batch 0 | Batch FPS: 24.56 | Latency: 40.72 ms/img
Batch 1 | Batch FPS: 24.41 | Latency: 40.97 ms/img

FINAL BATCH RESULTS
Avg FPS (throughput): 24.48
Avg Latency: 40.84 ms/image
Image | 70.62 ms | 14.16 FPS
Image | 69.91 ms | 14.30 FPS
Image | 69.33 ms | 14.42 FPS
Image | 53.33 ms | 18.75 FPS
Image | 54.32 ms | 18.41 FPS
Image | 52.73 ms | 18.96 FPS
Image | 52.98 ms | 18.88 FPS
Image | 53.03 ms | 18.86 FPS
Image | 52.65 ms | 18.99 FPS
Image | 53.86 ms | 18.57 FPS
Image | 53.09 ms | 18.84 FPS
Image | 54.46 ms | 18.36 FPS
Image | 53.89 ms | 18.56 FPS
Image | 54.25 ms | 18.43 FPS
Image | 54.52 ms | 18.34 FPS
Image | 53.74 ms | 18.61 FPS
Image | 53.62 ms | 18.65 FPS
Image | 55.10 ms | 18.15 FPS
Image | 53.82 ms | 18.58 FPS
Image | 53.72 ms | 18.61 FPS

FINAL RESULTS
Avg Latency: 56.15 ms
Throughput FPS: 17.81
